In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

# import FCI code
from causallearn.search.ConstraintBased.FCI import fci
from causallearn.utils.cit import fisherz
from causallearn.utils.GraphUtils import GraphUtils

#set project root as .../recidivism-causal
project_root = Path("/dcs/23/u2200504/thesis/recidivism-causal").resolve()

#set causal pitfalls root
cp_root = project_root / "data" / "raw" / "CausalPitfallsData"
#test it works
cp_root

output_dir = project_root/"results"/"graphs_fci"
output_dir.mkdir(parents=True,exist_ok=True)

In [2]:
#path to dataset
csv_path = cp_root/"berkson_paradox"/"admission_bias.csv"

#load into df and sanity check form
df = pd.read_csv(csv_path)
df.head(), df.shape

(   TestHigh  ExtraHigh
 0         1          1
 1         0          1
 2         1          0
 3         1          1
 4         0          1,
 (380, 2))

In [15]:
#scoring utility function given two adjacency matricies
def score_graph(W_est, W_true, labels_est, labels_true):
    #number of variables
    p = W_est.shape[0]

    #node orderings
    labels_est = list(labels_est)
    labels_true = list(labels_true)
    common = sorted(set(labels_est) & set(labels_true))

    #map ids
    idx_est = [labels_est.index(v) for v in common]
    idx_true = [labels_true.index(v) for v in common]

    W_est_aligned = np.asarray(W_est)[np.ix_(idx_est, idx_est)]
    W_true_aligned = np.asarray(W_true)[np.ix_(idx_true, idx_true)]

    est = (W_est!=0).astype(int)
    true = (W_true !=0).astype(int)

    #true positive, false positive, false negative, true negative
    tp = np.sum((est==1) & (true==1))
    fp = np.sum((est==1) & (true == 0))
    fn = np.sum((est==0) & (true == 1))
    tn = np.sum((est==0) & (true == 0))

    #structural hamming distance, true positive rate, false positive rate
    shd = fp + fn
    tpr = tp/(tp+fn) if (tp+fn) > 0 else np.nan
    fpr = fp/(fp+tn) if (fp + tn)>0 else np.nan
    fdr = fp/(tp+fp) if (tp+fp)>0 else np.nan

    #store scores in dictionary format for each 'experiment'
    return dict(TP=int(tp), FP=int(fp), FN=int(fn), TN=int(tn), SHD=int(shd), TPR = tpr, FPR = fpr, FDR= fdr)

In [43]:
def is_dir(W,i,j):
    # an edge is oriented i -> j if j has arrowhead (enc as 1) and i doesn't (anything but 1)
    return (W[i, j] != 1) and (W[j, i] == 1)

#pdag encoding -1 -> arrowtail, 1 -> arrowhead, 2 -> circle mark
def score_pdag(W_pdag, W_true, labels_pdag, labels_true):
    W_pdag = np.asarray(W_pdag)
    W_true = np.asarray(W_true)

    #skeleton graphs ignore directionality, score presence of edge
    S_est = ((W_pdag != 0) | (W_pdag.T != 0)).astype(int)
    S_true = ((W_true != 0) | (W_true.T != 0)).astype(int)

    s_metrics = score_graph(S_est,S_true, labels_pdag, labels_true) #(is it right that we should ignore false pos and false neg?)

    #orientation metric, score directed edges
    #+rearrange adjacency matrix if required
    
    #node orderings
    labels_pdag = list(labels_pdag)
    labels_true = list(labels_true)
    common = sorted(set(labels_pdag) & set(labels_true))

    #map ids
    idx_est = [labels_pdag.index(v) for v in common]
    idx_true = [labels_true.index(v) for v in common]

    W_pdag = np.asarray(W_pdag)[np.ix_(idx_est, idx_est)]
    W_true = np.asarray(W_true)[np.ix_(idx_true, idx_true)]

    #number of variables
    p = W_true.shape[0]
    tp_dir = fp_dir=fn_dir = tn_dir=0

    for i in range(p):
        for j in range(p):
            if i ==j:
                #print("i was equal j")
                continue

            true_ij = W_true[i,j]
            true_ji = W_true[j,i]
            est_ij = W_pdag[i,j]
            est_ji= W_pdag[j,i]

            #consider only pairs that are adjacent in ground truth DAG
            if(true_ij==0) and (true_ji==0):
                #print("wasnt adjacent in gt")
                continue

            # set correct direction according to the ground truth
            if true_ij == 1:
                true_dir = (i,j)
            else:
                true_dir = (j,i)

            #score direction according to the directed edge in the PDAG
            if is_dir(W_pdag, i, j):
                est_dir = (i, j)
            elif is_dir(W_pdag, j, i):
                est_dir = (j, i)
            elif (W_pdag[i, j] != 0) or (W_pdag[j, i] != 0):
                est_dir = None   # edge not oriented
            else:
                est_dir = None   # no edge

            if est_dir is None:
                # adjacency handled by skeleton metrics
                fn_dir += 1   # methodological choice: can choose to count lack of orientation as FN..
            elif est_dir == true_dir:
                tp_dir += 1
            else:
                fp_dir += 1

    shd_dir = fp_dir + fn_dir
    tpr_dir = tp_dir/(tp_dir+fn_dir) if (tp_dir+fn_dir) > 0 else np.nan
    fpr_dir = fp_dir/(fp_dir+tn_dir) if (fp_dir+tn_dir) > 0 else np.nan
    fdr_dir = fp_dir/(tp_dir+fp_dir) if (tp_dir+fp_dir) > 0 else np.nan

    orient_metrics = dict(
        TP=int(tp_dir), FP=int(fp_dir), FN=int(fn_dir), TN=int(tn_dir),
        SHD=int(shd_dir), TPR=tpr_dir, FPR=fpr_dir, FDR=fdr_dir
    )

    #return metric dictionaries, prefix with skeleton or orientation
    return {
        **{f"skel_{k}": v for k, v in s_metrics.items()},
        **{f"orient_{k}": v for k, v in orient_metrics.items()},
    }


In [38]:
# loop over datasets
csv_files = sorted(cp_root.rglob("*.csv"))
results = []

for csv_path in csv_files:
    #relative path
    rel = csv_path.relative_to(cp_root)

    #we only examine those with known ground truths i.e with _truth
    if csv_path.stem.endswith("_truth"):
        continue

    #expected ground truth path
    truth_path = csv_path.with_name(csv_path.stem + "_truth.csv")
    if not truth_path.exists():
        print("  Skipped (no ground-truth file)")
        continue

    print(f"Processing: {rel}")

    try:
        df = pd.read_csv(csv_path)
        if df.empty:
            print("  Skipped (empty file)")
            continue

        # Run FCI on this dataset
        data = df.to_numpy(dtype=float)
        g, _ = fci(data, fisherz, alpha=0.05) #causal graph object

        #extract adjacency matrix from PDAG
        W_pdag = g.graph
        print(W_pdag)

        #Output schema scenario__file__fci.png
        parts = rel.parts           
        scenario = parts[0] if len(parts) > 1 else "root"
        name_no_ext = csv_path.stem

        out_name = f"{scenario}__{name_no_ext}__FCI.png"
        out_path = output_dir / out_name

        # Save PNG using GraphUtils
        py_dot = GraphUtils.to_pydot(g, labels=list(df.columns))
        py_dot.write_png(str(out_path))

        #load ground truth and score
        gt_df = pd.read_csv(truth_path)
        gt_df.index.name = None
        gt_df.columns.name = None

        W_true = gt_df.to_numpy()
            
        # compute scores
        metrics = score_pdag(W_pdag, W_true, labels_pdag=list(df.columns), labels_true=list(gt_df.columns))
        print("metrics computed")
        print(metrics)
        metrics.update(
            dict(
                scenario=scenario,
                dataset=name_no_ext,
                algo="fci"
            )
        )
        results.append(metrics)
        
    except Exception as e:
        print(f"  ERROR on {rel}: {e}")

scores_df = pd.DataFrame(results)

  Skipped (no ground-truth file)
  Skipped (no ground-truth file)
  Skipped (no ground-truth file)
  Skipped (no ground-truth file)
  Skipped (no ground-truth file)
  Skipped (no ground-truth file)
  Skipped (no ground-truth file)
  Skipped (no ground-truth file)
  Skipped (no ground-truth file)
  Skipped (no ground-truth file)
  Skipped (no ground-truth file)
  Skipped (no ground-truth file)
  Skipped (no ground-truth file)
  Skipped (no ground-truth file)
  Skipped (no ground-truth file)
Processing: casual_effect/device_failure_data.csv


  0%|          | 0/7 [00:00<?, ?it/s]

[[0 0 2 0 2 2 0]
 [0 0 2 0 0 0 0]
 [1 1 0 1 1 0 0]
 [0 0 1 0 0 1 0]
 [2 0 2 0 0 2 0]
 [1 0 0 1 1 0 0]
 [0 0 0 0 0 0 0]]
metrics computed
{'skel_TP': 10, 'skel_FP': 6, 'skel_FN': 8, 'skel_TN': 25, 'skel_SHD': 14, 'skel_TPR': 0.5555555555555556, 'skel_FPR': 0.1935483870967742, 'skel_FDR': 0.375, 'orient_TP': 4, 'orient_FP': 2, 'orient_FN': 12, 'orient_TN': 0, 'orient_SHD': 14, 'orient_TPR': 0.25, 'orient_FPR': 1.0, 'orient_FDR': 0.3333333333333333}
  Skipped (no ground-truth file)
  Skipped (no ground-truth file)
  Skipped (no ground-truth file)
Processing: casual_effect/student_tutoring_data.csv


  0%|          | 0/7 [00:00<?, ?it/s]

X1 --> X2
X1 --> X3
X3 --> X2
[[ 0 -1 -1  1  1  0  0]
 [ 1  0  1  0  0  0  0]
 [ 1 -1  0  0  1  0  0]
 [ 2  0  0  0  0  0  0]
 [ 2  0 -1  0  0  2  0]
 [ 0  0  0  0  2  0  0]
 [ 0  0  0  0  0  0  0]]
metrics computed
{'skel_TP': 6, 'skel_FP': 8, 'skel_FN': 12, 'skel_TN': 23, 'skel_SHD': 20, 'skel_TPR': 0.3333333333333333, 'skel_FPR': 0.25806451612903225, 'skel_FDR': 0.5714285714285714, 'orient_TP': 0, 'orient_FP': 4, 'orient_FN': 14, 'orient_TN': 0, 'orient_SHD': 18, 'orient_TPR': 0.0, 'orient_FPR': 1.0, 'orient_FDR': 1.0}
  Skipped (no ground-truth file)
  Skipped (no ground-truth file)
  Skipped (no ground-truth file)
  Skipped (no ground-truth file)
  Skipped (no ground-truth file)
  Skipped (no ground-truth file)
  Skipped (no ground-truth file)
Processing: causal_direction_iv/clinical_trial_sem.csv


...

[[0 2 0 2 0 0]
 [2 0 0 0 0 0]
 [0 0 0 2 2 0]
 [1 0 1 0 1 0]
 [0 0 2 2 0 0]
 [0 0 0 0 0 0]]
metrics computed
{'skel_TP': 8, 'skel_FP': 2, 'skel_FN': 10, 'skel_TN': 16, 'skel_SHD': 12, 'skel_TPR': 0.4444444444444444, 'skel_FPR': 0.1111111111111111, 'skel_FDR': 0.2, 'orient_TP': 4, 'orient_FP': 2, 'orient_FN': 12, 'orient_TN': 0, 'orient_SHD': 14, 'orient_TPR': 0.25, 'orient_FPR': 1.0, 'orient_FDR': 0.3333333333333333}
Processing: treatment_mediator/job_training_policy.csv


  0%|          | 0/6 [00:00<?, ?it/s]

X4 --> X5
[[ 0  2  0  2  0  0]
 [ 2  0  0  0  0  0]
 [ 0  0  0  2 -1  0]
 [ 1  0  1  0 -1  0]
 [ 0  0  1  1  0  0]
 [ 0  0  0  0  0  0]]
metrics computed
{'skel_TP': 8, 'skel_FP': 2, 'skel_FN': 10, 'skel_TN': 16, 'skel_SHD': 12, 'skel_TPR': 0.4444444444444444, 'skel_FPR': 0.1111111111111111, 'skel_FDR': 0.2, 'orient_TP': 8, 'orient_FP': 0, 'orient_FN': 10, 'orient_TN': 0, 'orient_SHD': 10, 'orient_TPR': 0.4444444444444444, 'orient_FPR': nan, 'orient_FDR': 0.0}
Processing: treatment_mediator/tax_incentive_policy.csv


  0%|          | 0/6 [00:00<?, ?it/s]

[[0 2 0 2 0 0]
 [2 0 0 0 0 0]
 [0 0 0 2 2 0]
 [1 0 1 0 1 0]
 [0 0 2 2 0 0]
 [0 0 0 0 0 0]]
metrics computed
{'skel_TP': 8, 'skel_FP': 2, 'skel_FN': 10, 'skel_TN': 16, 'skel_SHD': 12, 'skel_TPR': 0.4444444444444444, 'skel_FPR': 0.1111111111111111, 'skel_FDR': 0.2, 'orient_TP': 4, 'orient_FP': 2, 'orient_FN': 12, 'orient_TN': 0, 'orient_SHD': 14, 'orient_TPR': 0.25, 'orient_FPR': 1.0, 'orient_FDR': 0.3333333333333333}


In [39]:
scores_df

,skel_TP,skel_FP,skel_FN,skel_TN,skel_SHD,skel_TPR,skel_FPR,skel_FDR,orient_TP,orient_FP,orient_FN,orient_TN,orient_SHD,orient_TPR,orient_FPR,orient_FDR,scenario,dataset,algo
0,10,6,8,25,14,0.555556,0.193548,0.375000,4,2,12,0,14,0.250000,1.0,0.333333,casual_effect,device_failure_data,fci
1,6,8,12,23,20,0.333333,0.258065,0.571429,0,4,14,0,18,0.000000,1.0,1.000000,casual_effect,student_tutoring_data,fci
2,6,2,0,8,2,1.000000,0.200000,0.250000,4,0,2,0,2,0.666667,NaN,0.000000,causal_direction_iv,clinical_trial_sem,fci
3,4,0,2,10,2,0.666667,0.000000,0.000000,0,0,6,0,6,0.000000,NaN,NaN,causal_direction_iv,ecommerce_sem,fci
4,0,4,6,6,10,0.000000,0.400000,1.000000,0,0,6,0,6,0.000000,NaN,NaN,causal_direction_iv,environment_sem,fci
5,4,0,2,10,2,0.666667,0.000000,0.000000,0,0,6,0,6,0.000000,NaN,NaN,causal_direction_iv,marketing_sem,fci
6,6,0,2,8,2,0.750000,0.000000,0.000000,0,0,8,0,8,0.000000,NaN,NaN,counterfactual_reasoning,climate_impact_sem,fci
7,6,2,2,6,4,0.750000,0.250000,0.250000,4,2,2,0,4,0.666667,1.0,0.333333,counterfactual_reasoning,clinical_trial_sem,fci
8,6,2,2,6,4,0.750000,0.250000,0.250000,4,2,2,0,4,0.666667,1.0,0.333333,counterfactual_reasoning,education_performance_sem,fci
9,8,2,0,6,2,1.000000,0.250000,0.200000,2,2,4,0,6,0.333333,1.0,0.500000,counterfactual_reasoning,investment_outcome_sem,fci


In [40]:
scores_path = project_root / "results" / "scores"/"scores_fci.csv"
scores_path.parent.mkdir(parents=True, exist_ok=True)
scores_df.to_csv(scores_path, index=False)